# Training Data Exploration
This notebook explores the logical reasoning puzzle training set in `train.csv`.

## Load Dependencies and Configure Display

In [2]:
import numpy as np
import pandas as pd
import textwrap

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_columns", 50)
np.set_printoptions(edgeitems=3, linewidth=120)

## Read `train.csv` into a DataFrame

In [3]:
train_path = "train.csv"
df = pd.read_csv(train_path)

print("Rows, columns:", df.shape)
df.head()

Rows, columns: (9500, 3)


,id,prompt,answer
0,00066667,"In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority o...",10010111
1,000b53cf,"In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority o...",01000011
2,00189f6a,"In Alice's Wonderland, secret encryption rules are used on text. Here are some examples:\nucoov pwgtfyoqg vorq yrjjoe -> queen discovers near valley\npqrsfv pqorzg wvgwpo trgbjo -> dragon dreams i...",cat imagines book
3,001b24c4,"In Alice's Wonderland, numbers are secretly converted into a different numeral system. Some examples are given below:\n11 -> XI\n15 -> XV\n94 -> XCIV\n19 -> XIX\nNow, write the number 38 in the Wo...",XXXVIII
4,001c63cb,"In Alice's Wonderland, secret encryption rules are used on text. Here are some examples:\nwkgqa lsrqaq wneeke -> mouse chases mirror\nwkgqa nwrjnvaq nv brmrla -> mouse imagines in palace\nsrppae o...",wizard creates secret


## Inspect Schema and Basic Stats

In [4]:
df.info()

df.describe(include="all").transpose()

print("Columns:", df.columns.tolist())
print("Dtypes:")
print(df.dtypes)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9500 entries, 0 to 9499
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      9500 non-null   object
 1   prompt  9500 non-null   object
 2   answer  9500 non-null   object
dtypes: object(3)
memory usage: 222.8+ KB
Columns: ['id', 'prompt', 'answer']
Dtypes:
id        object
prompt    object
answer    object
dtype: object


## Preview Samples and Prompt Structure

In [5]:
df.sample(3, random_state=7)[["id", "prompt", "answer"]]

sample_prompt = df.loc[0, "prompt"]
print("Sample prompt (full):\n")
print(sample_prompt)

sections = [s.strip() for s in sample_prompt.split("\n\n") if s.strip()]
print("\nDetected sections:", len(sections))
for i, section in enumerate(sections[:5], start=1):
    print(f"\nSection {i}:")
    print(textwrap.fill(section, width=120))

Sample prompt (full):

In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.

Here are some examples of input -> output:
01010001 -> 11011101
00001001 -> 01101101
00010101 -> 01010101
11111111 -> 10000001
10011101 -> 01000101
00111011 -> 00001001
10111101 -> 00000101
00100110 -> 10110011

Now, determine the output for: 00110100

Detected sections: 3

Section 1:
In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves
operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.

Section 2:
Here are some examples of input -> output: 01010001 -> 11011101 00001001 -> 01101101 00010101 -> 01010101 11111111 ->
10000001 10011101 -> 01000101 00111011 -> 00001001 10111101 -> 00000101 00100110 -> 10110011

Section 3:
Now, determine the outp

## Check Missing Values and Duplicates

In [6]:
missing_counts = df.isna().sum().sort_values(ascending=False)
missing_counts

print("Duplicate ids:", df.duplicated(subset=["id"]).sum())
print("Duplicate prompts:", df.duplicated(subset=["prompt"]).sum())

Duplicate ids: 0
Duplicate prompts: 0


## Analyze Prompt Lengths and Answer Distributions

In [7]:
prompt_lengths = df["prompt"].astype(str).str.len()
prompt_word_counts = df["prompt"].astype(str).str.split().str.len()

print("Prompt length stats (chars):")
print(prompt_lengths.describe())

print("\nPrompt length stats (words):")
print(prompt_word_counts.describe())

if "answer" in df.columns:
    answer_counts = df["answer"].value_counts()
    print("\nTop 10 answers:")
    print(answer_counts.head(10))
    if answer_counts.size <= 50:
        answer_counts.plot(kind="bar", figsize=(10, 4), title="Answer Frequency")

Prompt length stats (chars):
count    9500.000000
mean      301.515895
std       104.265311
min       177.000000
25%       209.000000
50%       281.000000
75%       371.000000
max       510.000000
Name: prompt, dtype: float64

Prompt length stats (words):
count    9500.000000
mean       50.042947
std        14.305601
min        32.000000
25%        36.000000
50%        46.000000
75%        65.000000
max        78.000000
Name: prompt, dtype: float64

Top 10 answers:
answer
00000000    103
11111111     32
00000001     30
LXXIII       25
XLIX         24
VI           24
XVII         24
LII          24
XIII         23
00000010     22
Name: count, dtype: int64


## Lookup Prompt and Answer by Id

In [8]:
def show_prompt_answer(row_id) -> None:
    """Print the full prompt and answer for a given id (string or number)."""
    if "id" not in df.columns:
        raise ValueError("Column 'id' not found in dataframe.")
    if "answer" not in df.columns:
        raise ValueError("Column 'answer' not found in dataframe.")

    id_series = df["id"].astype(str)
    row_id_str = str(row_id)
    match = df.loc[id_series == row_id_str]
    if match.empty:
        print(f"No row found for id={row_id_str}")
        return

    row = match.iloc[0]
    print(f"id: {row['id']}")
    print("\nPrompt:\n")
    print(row["prompt"])
    print("\nAnswer:\n")
    print(row["answer"])


def show_prompt_answer_by_index(n: int) -> None:
    """Print the full prompt and answer by 1-based row number."""
    if n < 1 or n > len(df):
        print(f"Index out of range. Use 1..{len(df)}")
        return

    row = df.iloc[n - 1]
    print(f"row: {n}  id: {row['id']}")
    print("\nPrompt:\n")
    print(row["prompt"])
    print("\nAnswer:\n")
    print(row["answer"])


# Example usage
# show_prompt_answer("abc123")
# show_prompt_answer_by_index(1)

In [21]:
show_prompt_answer_by_index(9000)

row: 9000  id: f2b23d11

Prompt:

In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.

Here are some examples of input -> output:
10111010 -> 01000000
10010101 -> 10100000
01011000 -> 00000000
11000111 -> 00100000
10001110 -> 01000000
11110000 -> 00000000
01100101 -> 00100000
00011011 -> 01000000

Now, determine the output for: 01111100

Answer:

10000000
